# Create the 54 parasha JSON datasets

Tony Jurg · Parasha Gamatria Browser · version 0.1.0 · 3 October 2026

Run from the repository root using an environment with `requirements.txt` installed. This notebook reads existing Text-Fabric values and exports a static dataset; it does not recalculate gematria.

## Contents <a class="anchor" id="TOC"></a>

* [1 — Setup](#bullet1)
* [2 — Load and verify the sources](#bullet2)
* [3 — Inspect parasha boundaries](#bullet3)
  * [Inspect English glosses](#glosses)
* [4 — Export all 54 files](#bullet4)
* [5 — Validate the result](#bullet5)
* [6 — Preview and attribution](#bullet6)


## 1 — Setup <a class="anchor" id="bullet1"></a>
[Back to ToC](#TOC)

BHSA TF files are fetched freshly from the pinned ETCBC/bhsa GitHub revision on each run; an internet connection is required. The paths below refer only to local `gematria_TF` and `BHSaddons` repositories beside this one. Override those paths if needed. No GitHub credentials are embedded or required by the exported site.


In [ ]:
from pathlib import Path
from export_parashot import ROOT, default_sources, load_sources, partition_verses, export_all, validate_document, digest
from data_codec import expand_document
import json

sources = default_sources()
sources


## 2 — Load and verify the sources <a class="anchor" id="bullet2"></a>
[Back to ToC](#TOC)

Checksums and revisions come from `source-lock.json`. BHSA is downloaded directly from ETCBC into a temporary directory and every file is checksum-verified before loading. Existing BHSA caches are never used. The fresh files remain available while the loaded API is in use; they and compiled BHSA data are removed when the API is released or the process exits. Local gematria_TF and BHSaddons files are also verified; missing or changed features stop the export.


In [ ]:
api, lock = load_sources(sources)
for name, source in lock.items():
    print(name, source['commit'], source['version'])


## 3 — Inspect parasha boundaries <a class="anchor" id="bullet3"></a>
[Back to ToC](#TOC)

Portion boundaries and names come from BHSaddons, not a handwritten range list. Validate that all five Torah books are covered exactly once. Some source verse counters are absent; navigation follows BHSA text order and the export records an audit.


In [ ]:
groups = partition_verses(api)
assert len(groups) == 54
for number in (1, 13, 54):
    verses = groups[number]
    print(number, api.F.parashatrans.v(verses[0]),
          api.T.sectionFromNode(verses[0]), api.T.sectionFromNode(verses[-1]), len(verses))


### Inspect English glosses <a id="glosses"></a>
[Back to ToC](#TOC)

The pinned BHSA `gloss.tf` supplies English lexical glosses. Each decoded word contains `gloss` (or `null` when absent). These are dictionary meanings, not verse translations. Enable **Show glosses** in the browser to display them alongside Hebrew words in the reader and matching passages.


In [ ]:
for word in api.L.d(groups[1][0], otype='word'):
    print(word, api.F.g_word_utf8.v(word), api.F.gloss.v(word))


## 4 — Export all 54 files <a class="anchor" id="bullet4"></a>
[Back to ToC](#TOC)

This writes compact schema-2 `site/data/01.json` through `54.json` plus `catalogue.js`. Rerunning replaces those generated outputs. Shared string, form and value tables remove duplication without dropping data. Use `expand_document` to obtain readable word records. Each word carries five gematria methods for dictionary, written/read word-part, and written/read whole-word representations. Sentences, clauses and phrases preserve explicit BHSA word membership.


In [ ]:
output = ROOT / 'site' / 'data'
manifest = export_all(api, lock, output)
print('JSON files:', len(list(output.glob('*.json'))))
print('Total word units:', sum(p['words'] for p in manifest['parashot']))
print('Source verse-counter audit:', manifest['sourceVerseCounterAudit'])


## 5 — Validate the result <a class="anchor" id="bullet5"></a>
[Back to ToC](#TOC)

Check every JSON file against its checksum and schema constraints, plus a known value from Genesis 1:1. Run the Python and JavaScript test suites described in the README for further checks.


In [ ]:
assert len(manifest['parashot']) == len(list(output.glob('*.json'))) == 54
for entry in manifest['parashot']:
    path = output / entry['file']
    assert digest(path) == entry['sha256']
    document = expand_document(json.loads(path.read_text(encoding='utf-8')))
    assert validate_document(document)
    assert all('gloss' in word for word in document['words'])
assert api.F.gem_hechrechi_full_ketiv_ident.v(1) == 913
assert sum(p['words'] for p in manifest['parashot']) == 112927
print('All 54 files validated. Bereshit: בראשית = 913.')


## 6 — Preview and attribution <a class="anchor" id="bullet6"></a>
[Back to ToC](#TOC)

From a terminal in this repository, run `python -m http.server 8767 --bind 127.0.0.1 --directory site`, then open http://127.0.0.1:8767/ . The browser uses static HTML, CSS, JavaScript and the generated JSON files.

Sources: [ETCBC BHSA](https://etcbc.github.io/bhsa/) (CC BY-NC 4.0), [Tony Jurg’s gematria_TF](https://github.com/tonyjurg/gematria_TF) (data: CC BY-NC 4.0), and [BHSaddons](https://github.com/tonyjurg/BHSaddons) (repository: CC BY 4.0; consumed TF headers: CC BY-NC 4.0). Exported data follows CC BY-NC 4.0; new code is MIT; notebook prose is CC BY 4.0. See README and LICENSE-DATA.md.

The clickable contents pattern follows Tony Jurg’s [BHSA TF-IDF notebook](https://github.com/tonyjurg/Create_TF-IDF_Text-Fabric_features/blob/main/create_tfidf_features.ipynb).
